# Agent-to-Agent / A2A Security and Delegation

Secure a realistic support-triage handoff from a coordinator to an A2A peer. The model may propose the task; trusted code authenticates peers, attenuates authority, executes policy, and verifies the effect.

**Safety boundary:** this notebook is deterministic, offline, and credential-free. It uses training keys, supplied DNS answers, and in-memory state. It never contacts an agent, authorization server, webhook, ticket system, or queue.

## Objectives

You will inspect official A2A objects, bind a reviewed Agent Card to a workload identity, attenuate parent authority, enforce task/message/lifecycle invariants, make retries idempotent, block webhook SSRF, and measure adversarial outcomes.

In [ ]:
import importlib.metadata
import json
import runpy
from dataclasses import asdict, replace
from pathlib import Path
import jwt

here = Path.cwd()
if not (here / 'lab.py').exists():
    here = here / 'curriculum/advanced/18-agent-to-agent-a2a-security-delegation'
lab = runpy.run_path(here / 'lab.py')
print('Loaded Course 18 lab')

In [ ]:
{'a2a_sdk': importlib.metadata.version('a2a-sdk'), 'protocol_target': lab['PROTOCOL_VERSION']}

## 1. Protocol objects are not authority

An Agent Card advertises interfaces and skills. A Message carries input. A Task represents server-owned work. None grants permission. Authentication and authorization remain application responsibilities.

In [ ]:
card = lab['build_agent_card']()
{
    'protobuf_type': card.DESCRIPTOR.full_name,
    'name': card.name,
    'version': card.version,
    'interface': card.supported_interfaces[0].url,
    'protocol': card.supported_interfaces[0].protocol_version,
    'skills': [skill.id for skill in card.skills],
    'digest': lab['card_digest'](card),
}

## 2. Keep identities separate

The human subject, caller workload, recipient workload, tenant, and resource owner are different principals. Message metadata such as `tenantHint` and `authorization` remains untrusted even when it looks authoritative.

In [ ]:
env = lab['reviewed_environment']()
{
    'human_subject': env.parent.subject,
    'caller_agent': env.sender.agent_id,
    'caller_workload': env.sender.spiffe_id,
    'recipient_agent': env.server.agent_id,
    'tenant': env.request.tenant_id,
    'resource': env.request.resource,
}

In [ ]:
{
    'peer': env.peer_record.agent_id,
    'workload': env.peer_record.workload_id,
    'tenants': env.peer_record.tenants,
    'skills': env.peer_record.skills,
    'extensions': env.peer_record.supported_extensions,
    'state': env.peer_record.state,
    'reviewed_card_digest': env.peer_record.card_digest,
}

## 3. Attenuate authority at the handoff

The broker intersects the parent with the proposed task. The child is bound to subject, actor, recipient, tenant, action, resource, purpose, IDs, protocol/card version, extension set, chain, and a maximum five-minute lifetime.

In [ ]:
claims = jwt.decode(env.token, options={'verify_signature': False})
{key: claims[key] for key in [
    'sub', 'act', 'aud', 'tenant', 'action', 'resource', 'purpose',
    'message_id', 'operation_id', 'context_id', 'depth', 'chain',
]}

In [ ]:
escalated = replace(env.request, action='ticket.delete')
try:
    env.broker.issue(
        parent=env.parent, sender=env.sender, recipient_agent_id='support-agent',
        request=escalated, now=env.now,
    )
except lab['A2ADenied'] as exc:
    print('Denied:', exc.code)

## 4. Bind the wire message to the grant

The recipient compares authenticated state and token claims with the message and server-side request. It also verifies the reviewed card, skill, protocol version, and required extension.

In [ ]:
{
    'protobuf_type': env.message.DESCRIPTOR.full_name,
    'message_id': env.message.message_id,
    'context_id': env.message.context_id,
    'client_supplied_task_id': env.message.task_id or None,
    'extensions': list(env.message.extensions),
    'untrusted_metadata': dict(env.message.metadata),
}

In [ ]:
task, result = env.server.receive(
    principal=env.sender, request=env.request, message=env.message,
    task_grant=env.token, now=env.now, push_ips=('93.184.216.34',),
)
{
    'task_id': task.id,
    'context_id': task.context_id,
    'state': lab['TaskState'].Name(task.status.state),
    'result': result,
}

## 5. Enforce a task state machine

The server creates task IDs. `submitted → working → completed` is legal; jumping from submitted to completed or reviving a terminal task is not. Input/auth-required are interrupted states that can resume or cancel.

In [ ]:
probe = lab['Task'](
    id='task-probe', context_id='ctx-probe',
    status=lab['TaskStatus'](state=lab['TaskState'].TASK_STATE_SUBMITTED),
)
working = lab['transition'](probe, lab['TaskState'].TASK_STATE_WORKING)
try:
    lab['transition'](probe, lab['TaskState'].TASK_STATE_COMPLETED)
except lab['A2ADenied'] as exc:
    print('Legal state:', lab['TaskState'].Name(working.status.state), '| illegal jump:', exc.code)

## 6. Treat timeouts as unknown outcomes

Retry the exact message with the same message and operation IDs. The task store returns the prior task/result, while the resource-side effect ledger proves the external effect happened once.

In [ ]:
retried_task, retried_result = env.server.receive(
    principal=env.sender, request=env.request, message=env.message,
    task_grant=env.token, now=env.now, push_ips=('93.184.216.34',),
)
{
    'same_task': retried_task.id == task.id,
    'same_result': retried_result == result,
    'external_effect_count': len(env.resource.effect_ledger),
}

## 7. Push callbacks create an SSRF boundary

The offline validator requires an allow-listed HTTPS host and supplied global IP addresses. Production must repeat validation during DNS resolution, redirects, and connection establishment.

In [ ]:
approved = lab['validate_push_url'](
    'https://hooks.acme.test/a2a/tasks', resolved_ips=('93.184.216.34',)
)
try:
    lab['validate_push_url'](
        'https://hooks.acme.test/latest/meta-data', resolved_ips=('169.254.169.254',)
    )
except lab['A2ADenied'] as exc:
    print('Approved:', approved, '| metadata target:', exc.code)

## 8. Re-authorize the final effect

The ticket service does not trust the coordinator or recipient merely because the A2A hop succeeded. It checks its own audience, allowed action, tenant ownership, and operation ID.

In [ ]:
verified = lab['verify_task_grant'](
    env.token, recipient_agent_id='support-agent', sender=env.sender,
    request=env.request, now=env.now,
)
try:
    env.resource.execute({**verified, 'aud': 'coordinator'}, env.request)
except lab['A2ADenied'] as exc:
    print('Independent denial:', exc.code)

## 9. Evaluate adversarial outcomes

The campaign includes one legitimate handoff plus tenant, action, resource, workload, card, extension, push-SSRF, and message-binding failures. Unsafe allow rate is the primary security metric.

In [ ]:
campaign = lab['run_campaign']()
campaign['results']

In [ ]:
campaign['metrics']

## 10. Check architecture regret

A2A adds a peer directory, token service, task store, notification delivery, more policy checks, and more partial failures. Use it when independent agents, long-running collaboration, or interoperability justify those costs. Prefer a direct authorized service call when they do not.

In [ ]:
demo = lab['run_demo']()
{
    'direct_baseline_policy_checks': demo['architecture']['direct_baseline_policy_checks'],
    'a2a_boundary_checks': demo['architecture']['a2a_boundary_checks'],
    'extra_failure_domains': demo['architecture']['extra_failure_domains'],
    'effect_count': demo['effect_count'],
    'unsafe_allow_rate': demo['campaign']['metrics']['unsafe_allow_rate'],
}

## Exercises and Reflection

1. Add an approval receipt for `ticket.comment` and prove the child remains a subset of the parent.
2. Add a second hop, then test chain uniqueness, depth, lifetime, and resource attenuation.
3. Design a transactional dedupe table and show where the external-effect commit boundary belongs.
4. Add task get/cancel authorization without leaking whether another tenant's task exists.
5. Explain which requirement truly needs A2A here. Would a direct service call reduce risk?

**Exit question:** why must the final ticket service reject a wrong audience even after the A2A recipient accepted the task?